## 섹션 0-1: 환경 감지 및 경로 설정

### 팀원 로컬 실행 방법
1. `git clone https://github.com/ehdgus4173/DACON_Structure_Stability.git`
2. 레포 상위 폴더에 `dataset/` 폴더 생성 후 데이콘 데이터 배치:
   ```
   DACON_Structure_Stability/  ← 레포
   dataset/                    ← 데이터 (레포와 같은 위치)
     train/
     dev/
     test/
     train.csv
     dev.csv
     sample_submission.csv
   ```
3. conda 환경 생성: `conda create -n stability python=3.10`
4. 패키지 설치: `pip install -r requirements.txt`
5. Jupyter 커널 등록: `python -m ipykernel install --user --name stability`
6. 이 노트북 실행

In [ ]:
import os, sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    DATA_DIR    = '/kaggle/input/datasets/junseopkim/structure-stability-data/data'
    SRC_DIR     = '/kaggle/input/datasets/junseopkim/structure-stability-src/src'
    OUT_DIR     = '/kaggle/working/outputs'
    IMG_SIZE    = 384
    NUM_WORKERS = 2
else:
    # config.py 기반 — 레포 어느 팀원 PC에서든 자동으로 경로 적용
    REPO_ROOT = Path('..').resolve()
    sys.path.insert(0, str(REPO_ROOT))
    from config import DATASET_DIR, CHECKPOINT_DIR, PROJECT_ROOT, PHYS_COLS_V2
    DATA_DIR    = str(DATASET_DIR)
    SRC_DIR     = str(REPO_ROOT / 'src')
    OUT_DIR     = str(REPO_ROOT / 'outputs')
    IMG_SIZE    = 224
    NUM_WORKERS = 0

sys.path.append(SRC_DIR)

FIG_DIR = f"{OUT_DIR}/../reports/figures" if not IS_KAGGLE else f"{OUT_DIR}/figures"
os.makedirs(f'{OUT_DIR}/models', exist_ok=True)
os.makedirs(f'{OUT_DIR}/logs', exist_ok=True)
os.makedirs(f'{OUT_DIR}/submissions', exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"환경: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"IMG_SIZE: {IMG_SIZE}")
print(f"GPU: {os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()}")

## 섹션 0-2: import 및 seed 고정

In [ ]:
from model import MultiViewNet
from experiment_utils import set_seed, EarlyStopping, run_experiment, run_inference, train_one_epoch, evaluate

import random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss, accuracy_score
from dataset import build_datasets, build_test_dataset
from augmentation import get_train_transform, get_val_transform

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 섹션 1-2: 구조 확인

In [ ]:
test_model = MultiViewNet(backbone_name='resnet18', fusion_mode='concat', img_size=IMG_SIZE).to(DEVICE)
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out = test_model(dummy, dummy)
print(f"출력 shape: {out.shape}")
print(f"총 파라미터: {sum(p.numel() for p in test_model.parameters()):,}")
del test_model, dummy

## 섹션 2: EXP-001 (HIGH 증강)

In [ ]:
config_exp001 = {
    "exp_id": "001", "model_version": "2", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "concat",
    "use_physics": False, "use_physics_features": False, "physics_dim": 6,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "imagenet", "dev_split_ratio": 0.0, "img_size": IMG_SIZE,
    "aug_params": {
        "brightness_p": 0.7,
        "gamma_p": 0.5,
        "hsv_p": 0.5,
        "shift_scale_p": 0.4,
        "perspective_p": 0.4,
        "flip_p": 0.3
    },
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 5, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}
result_exp001 = run_experiment(config_exp001)

## 섹션 3: EXP-002 (Dev 학습 포함)

In [ ]:
config_exp002 = {**config_exp001, "exp_id": "002", "model_version": "3", "dev_split_ratio": 0.8}
result_exp002 = run_experiment(config_exp002)
print("⚠️ Val 샘플 20개 — EXP-001(Val 100개)과 직접 비교 주의")

## 섹션 4: EXP-004 (Custom 정규화)

In [ ]:
config_exp004 = {**config_exp001, "exp_id": "004", "model_version": "4", "norm_version": "custom"}
result_exp004 = run_experiment(config_exp004)

## 섹션 5: EXP-005 (EfficientNet-B0)

In [ ]:
test_eff = MultiViewNet(backbone_name='efficientnet_b0', fusion_mode='concat', img_size=IMG_SIZE).to(DEVICE)
dummy = torch.randn(16, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
_ = test_eff(dummy, dummy)
if torch.cuda.is_available():
    print(f"VRAM 사용량: {torch.cuda.memory_allocated()/(1024**2):.1f} MB")
del test_eff, dummy; torch.cuda.empty_cache()

config_exp005 = {**config_exp001, "exp_id": "005", "model_version": "5",
                 "model_name": "efficientnet", "backbone": "efficientnet_b0"}
result_exp005 = run_experiment(config_exp005)

## 섹션 6: EXP-006 (물리 피처 6개)

In [ ]:
config_exp006 = {**config_exp001, "exp_id": "006", "model_version": "6",
                 "use_physics": True, "use_physics_features": True, "physics_dim": 6}
result_exp006 = run_experiment(config_exp006)

## 섹션 7: EXP-007 (Fusion 구조 개선)

In [ ]:
config_exp007 = {**config_exp001, "exp_id": "007", "model_version": "7", "fusion_mode": "diff_concat"}
result_exp007 = run_experiment(config_exp007)

## 섹션 8: EXP-008 (최적 조합)
Custom 정규화 + diff_concat Fusion + Dev 포함 + 물리 피처 6개 + HIGH 증강
결과: Best Val LogLoss 0.1028 (Epoch 2) — Public 0.16137 / 175위

In [ ]:
config_exp008 = {
    "exp_id": "008", "model_version": "8", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "diff_concat",
    "use_physics": True, "use_physics_features": True, "physics_dim": 6,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "custom", "dev_split_ratio": 0.8, "img_size": IMG_SIZE,
    "aug_params": {
        "brightness_p": 0.7, "gamma_p": 0.5, "hsv_p": 0.5,
        "shift_scale_p": 0.4, "perspective_p": 0.4, "flip_p": 0.3
    },
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 7, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}
result_exp008 = run_experiment(config_exp008)

## 섹션 9: EXP-009~011 (dev_split_ratio Ablation)
ratio=0.0 / 0.5 / 1.0 비교
결과: ratio=0.5 (Val 0.1037) > ratio=0.0 (0.1880) > ratio=1.0 (0.5896)

In [ ]:
for ratio, exp_id, ver in [(0.0, '009', '9'), (0.5, '010', '10'), (1.0, '011', '11')]:
    cfg = {
        **config_exp008,
        "exp_id": exp_id, "model_version": ver,
        "dev_split_ratio": ratio,
        "epochs": 3 if ratio == 1.0 else 50,
        "early_stopping_patience": 7
    }
    result = run_experiment(cfg)
    print(f"ratio={ratio} → Best Val LogLoss: {result['best_val_logloss']:.4f}")

## 섹션 10: EXP-012 (최적 조합 + ratio=0.5)
EXP-008 조합에서 dev_split_ratio 0.8→0.5로 변경
결과: Best Val LogLoss 0.0608 (Epoch 19)

In [ ]:
config_exp012 = {
    **config_exp008,
    "exp_id": "012", "model_version": "12",
    "dev_split_ratio": 0.5,
    "early_stopping_patience": 10
}
result_exp012 = run_experiment(config_exp012)

## 섹션 11: 추론 — submission_v2.csv 생성 (EXP-008 기준)
Public LogLoss: 0.16137 / 175위

In [ ]:
preds_v2 = run_inference(config=config_exp008, model_path=result_exp008['model_path'])
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = preds_v2
sub_df['stable_prob']   = 1.0 - preds_v2
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v2.csv", index=False)
print(f"submission_v2.csv 저장 완료")
print(f"prob 범위: {preds_v2.min():.4f} ~ {preds_v2.max():.4f}")

## 섹션 13: EXP-013 (CosineAnnealingLR)
EXP-012 기반 — LR 스케줄러 추가
결과: Best Val LogLoss 0.0756 (Epoch 13)

In [ ]:
config_exp013 = {
    **config_exp012,
    "exp_id": "013", "model_version": "13",
    "lr_scheduler": "cosine", "lr_min": 1e-5,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp013 = run_experiment(config_exp013)

## 섹션 14: EXP-014 (EfficientNet-B4 백본)
EXP-012 기반 — 백본 교체. batch=8 제약으로 학습 불안정.
결과: Best Val LogLoss 0.3281 (Epoch 1) — gradient_accumulation 재시도 필요

In [ ]:
test_b4 = MultiViewNet(
    backbone_name='efficientnet_b4', fusion_mode='diff_concat',
    use_physics=True, physics_dim=6, img_size=IMG_SIZE
).to(DEVICE)
dummy = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
_ = test_b4(dummy, dummy, torch.randn(4, 6).to(DEVICE))
if torch.cuda.is_available():
    print(f"VRAM 사용량: {torch.cuda.memory_allocated()/(1024**2):.1f} MB")
del test_b4, dummy; torch.cuda.empty_cache()

config_exp014 = {
    **config_exp012,
    "exp_id": "014", "model_version": "14",
    "model_name": "efficientnet_b4", "backbone": "efficientnet_b4",
    "batch_size": 8, "lr_scheduler": "cosine", "lr_min": 1e-5,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp014 = run_experiment(config_exp014)

## 섹션 15: EXP-015 (Seed 앙상블)
seed 42/43/44로 3번 학습 후 test 확률값 평균
결과: seed42=0.0994, seed43=0.0491, seed44=0.0438 (전체 최고)
Public: 0.07936 / 100위 (submission_v4 앙상블+TempScale)

In [ ]:
seed_preds = []
seed_results = []

for seed in [42, 43, 44]:
    cfg = {
        **config_exp012,
        "exp_id": f"015_seed{seed}", "model_version": f"15s{seed}",
        "random_state": seed, "epochs": 50, "early_stopping_patience": 10
    }
    result = run_experiment(cfg)
    seed_results.append(result)
    preds = run_inference(config=cfg, model_path=result['model_path'])
    seed_preds.append(preds)
    print(f"seed={seed} → Best Val LogLoss: {result['best_val_logloss']:.4f}")

ensemble_preds = np.mean(seed_preds, axis=0)
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = ensemble_preds
sub_df['stable_prob']   = 1.0 - ensemble_preds
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v3_ensemble.csv", index=False)
print(f"\n앙상블 완료 | prob 범위: {ensemble_preds.min():.4f}~{ensemble_preds.max():.4f}")

## 섹션 16: EXP-016 (Phys MLP 구조)
물리 피처를 별도 MLP로 처리 (팀원 구조 도입)
결과: Best Val LogLoss 0.1328 (Epoch 5) — BatchNorm 초기 불안정

In [ ]:
config_exp016 = {
    **config_exp012,
    "exp_id": "016", "model_version": "16",
    "use_phys_mlp": True, "epochs": 50, "early_stopping_patience": 10
}
result_exp016 = run_experiment(config_exp016)

## 섹션 17: 추론 — submission_v3.csv (EXP-012 기준)

In [ ]:
preds_v3 = run_inference(config=config_exp012, model_path=result_exp012['model_path'])
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = preds_v3
sub_df['stable_prob']   = 1.0 - preds_v3
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v3.csv", index=False)
print(f"submission_v3.csv 저장 완료")
print(f"prob 범위: {preds_v3.min():.4f} ~ {preds_v3.max():.4f} | 평균: {preds_v3.mean():.4f}")

## 섹션 19: EXP-017 (팀원 피처 통합 — 20개)
EXP-012 기반 — physics_dim 6 → 20
팀원(임동현) 설계 피처:
- 신규 1세대: f_cy_ratio, t_frontback_mass_ratio, t_pa_cx_offset, t_pa_cy_offset
- 격자 시점 6개: front/top_grid_detected, tilt_angle, perspective_ratio
- 2세대: FS_overturning, kern_ratio, effective_eccentricity, eccentric_combined, p_delta_eccentricity
사용 전 extract_base.py + extract_advanced.py 실행으로 features/combined_features_v3.csv 생성 필요

In [ ]:
# 팀원 피처 추출 (최초 1회만 실행, 이후 skip)
import subprocess
features_path = f"{OUT_DIR}/../features/combined_features_v3.csv"
if not os.path.exists(features_path):
    print("피처 추출 시작...")
    subprocess.run(['python', f'{SRC_DIR}/features/extract_base.py'], check=True)
    subprocess.run(['python', f'{SRC_DIR}/features/extract_advanced.py'], check=True)
    print("피처 추출 완료")
else:
    print(f"피처 파일 이미 존재: {features_path}")

# PHYS_COLS_V2 피처 이름 확인
if not IS_KAGGLE:
    print(f"\n피처 수: {len(PHYS_COLS_V2)}개")
    for i, col in enumerate(PHYS_COLS_V2):
        print(f"  {i+1:2d}. {col}")

In [ ]:
# EXP-017: 팀원 피처 20개 + 최적 조합
config_exp017 = {
    **config_exp012,
    "exp_id": "017", "model_version": "17",
    "physics_dim": 20,          # 6 → 20 (팀원 피처 전체)
    "use_physics": True,
    "use_physics_features": True,
    "epochs": 50,
    "early_stopping_patience": 10
}
result_exp017 = run_experiment(config_exp017)

## 섹션 20: EXP-018 (팀원 피처 + Phys MLP)
EXP-017 기반 — 물리 피처 20개를 별도 MLP로 처리
physics_dim이 커지면 MLP 구조가 더 의미있어짐

In [ ]:
config_exp018 = {
    **config_exp017,
    "exp_id": "018", "model_version": "18",
    "use_phys_mlp": True,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp018 = run_experiment(config_exp018)

## 섹션 21: EXP-019 (격자 피처 Ablation)
팀원 피처 중 격자 기반 카메라 시점 피처(6개)만 제외한 실험
격자 피처의 독립 기여도 측정

In [ ]:
# 격자 피처 제외 (physics_dim=14)
# PHYS_COLS_V2에서 _grid_ 포함 피처 6개 제외
config_exp019 = {
    **config_exp012,
    "exp_id": "019", "model_version": "19",
    "physics_dim": 14,          # 20 - 6(grid 피처)
    "use_physics": True,
    "use_physics_features": True,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp019 = run_experiment(config_exp019)

## 섹션 22: EXP-020 (Seed 앙상블 — 팀원 피처 20개)
최고 성능 팀원 피처 config 기반 seed 앙상블

In [ ]:
# EXP-017, EXP-018 중 더 좋은 config 선택
# 아래 base_config를 결과에 따라 수정
best_phys_config = config_exp017  # 또는 config_exp018

seed_preds_v5 = []
for seed in [42, 43, 44]:
    cfg = {
        **best_phys_config,
        "exp_id": f"020_seed{seed}", "model_version": f"20s{seed}",
        "random_state": seed, "epochs": 50, "early_stopping_patience": 10
    }
    result = run_experiment(cfg)
    preds = run_inference(config=cfg, model_path=result['model_path'])
    seed_preds_v5.append(preds)
    print(f"seed={seed} → Best Val LogLoss: {result['best_val_logloss']:.4f}")

ensemble_v5 = np.mean(seed_preds_v5, axis=0)
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = np.clip(ensemble_v5, 1e-7, 1-1e-7)
sub_df['stable_prob']   = 1.0 - sub_df['unstable_prob']
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v5.csv", index=False)
print(f"\nsubmission_v5.csv 저장 완료")
print(f"prob 범위: {ensemble_v5.min():.4f}~{ensemble_v5.max():.4f}")

## 섹션 18: 전체 실험 결과 비교표

In [ ]:
results_all = [
    {"exp_id": "baseline", "model": "resnet18", "변경사항": "없음(기준점)",    "best_val_logloss": 1.5369, "best_epoch": 1,  "public": 1.67225},
    {"exp_id": "001",      "model": "resnet18", "변경사항": "HIGH 증강",        "best_val_logloss": 0.5007, "best_epoch": 1,  "public": None},
    {"exp_id": "002",      "model": "resnet18", "변경사항": "Dev 포함(0.8)",    "best_val_logloss": 0.3687, "best_epoch": 3,  "public": None},
    {"exp_id": "004",      "model": "resnet18", "변경사항": "Custom 정규화",     "best_val_logloss": 0.1700, "best_epoch": 11, "public": None},
    {"exp_id": "005",      "model": "efficientnet_b0", "변경사항": "EfficientNet-B0", "best_val_logloss": 0.5081, "best_epoch": 1, "public": None},
    {"exp_id": "006",      "model": "resnet18", "변경사항": "물리 피처 6개",      "best_val_logloss": 0.3438, "best_epoch": 12, "public": None},
    {"exp_id": "007",      "model": "resnet18", "변경사항": "diff_concat Fusion","best_val_logloss": 0.1996, "best_epoch": 7,  "public": None},
    {"exp_id": "008",      "model": "resnet18", "변경사항": "최적 조합",         "best_val_logloss": 0.1028, "best_epoch": 2,  "public": 0.16137},
    {"exp_id": "009",      "model": "resnet18", "변경사항": "ratio=0.0",        "best_val_logloss": 0.1880, "best_epoch": 7,  "public": None},
    {"exp_id": "010",      "model": "resnet18", "변경사항": "ratio=0.5",        "best_val_logloss": 0.1037, "best_epoch": 22, "public": None},
    {"exp_id": "011",      "model": "resnet18", "변경사항": "ratio=1.0",        "best_val_logloss": 0.5896, "best_epoch": 1,  "public": None},
    {"exp_id": "012",      "model": "resnet18", "변경사항": "최적조합+ratio=0.5","best_val_logloss": 0.0608, "best_epoch": 19, "public": None},
    {"exp_id": "013",      "model": "resnet18", "변경사항": "CosineAnnealingLR","best_val_logloss": 0.0756, "best_epoch": 13, "public": None},
    {"exp_id": "014",      "model": "efficientnet_b4", "변경사항": "EfficientNet-B4", "best_val_logloss": 0.3281, "best_epoch": 1, "public": None},
    {"exp_id": "015_s42",  "model": "resnet18", "변경사항": "Seed42",           "best_val_logloss": 0.0994, "best_epoch": 18, "public": None},
    {"exp_id": "015_s43",  "model": "resnet18", "변경사항": "Seed43",           "best_val_logloss": 0.0491, "best_epoch": 26, "public": None},
    {"exp_id": "015_s44",  "model": "resnet18", "변경사항": "Seed44",           "best_val_logloss": 0.0438, "best_epoch": 25, "public": None},
    {"exp_id": "016",      "model": "resnet18", "변경사항": "Phys MLP",         "best_val_logloss": 0.1328, "best_epoch": 5,  "public": None},
    {"exp_id": "v4",       "model": "ensemble", "변경사항": "앙상블+TempScale",  "best_val_logloss": None,   "best_epoch": None, "public": 0.07936},
]
result_df = pd.DataFrame(results_all).sort_values('best_val_logloss')
print(result_df.to_string(index=False))
result_df.to_csv(f"{OUT_DIR}/logs/experiment_summary.csv", index=False)